# 模块C：智能客服与舆情分析（30分）

基于社交媒体评论数据，完成 **文本预处理 → TF-IDF特征 → SVM情感分类 → 评估指标 → LSTM情感分析 → K-Means话题聚类 → 肘部法则选K** 全流程。

| 任务 | 内容 | 分值 |
| --- | --- | --- |
| C1 | 中文文本预处理 | 2 |
| C2 | TF-IDF 特征提取 | 2 |
| C3 | 传统机器学习情感分类(SVM) | 3 |
| C4 | 分类评估指标计算 | 3 |
| C5 | 文本序列化与填充 | 4 |
| C6 | LSTM情感分析模型构建 | 4 |
| C7 | LSTM模型训练与评估 | 6 |
| C8 | K-Means热点话题聚类 | 3 |
| C9 | 肘部法则确定最佳聚类数 | 3 |

> 使用方法：从上到下按顺序运行；每个任务的库在各自 cell 内单独导入，方便单独记忆与复习。
> 标签约定：**0=负面，1=中性，2=正面**。

## C1 中文文本预处理（2分）
**做什么**：jieba精确分词 → 去停用词和长度<2的词 → 空格连接 → 打印前5条前后对比。

**记忆点**：`jieba.cut` 分词；列表推导式过滤；`' '.join` 拼接。

In [6]:
# ===== C1：中文文本预处理 =====
import os
import jieba

# 示例评论数据(比赛时替换为赛题CSV中的 text/label 列),每类10条共30条
texts = [
    # 负面 0
    "这个产品质量太差了,用了两天就坏,非常失望",
    "屏幕有划痕,客服推诿不处理,差评",
    "物流慢得要死,等了一周才到,包装还是破的",
    "完全不值这个价格,做工粗糙,以后再也不买了",
    "电池续航很差,半天就没电,卡顿严重",
    "发错货了,申请退款一直没人理,太坑了",
    "味道刺鼻,用着过敏,假货无疑",
    "噪音大得离谱,根本没法用,后悔死了",
    "虚假宣传,功能和描述完全不一样,骗子",
    "用一次就坏了,质量堪忧,强烈不推荐",
    # 中性 1
    "一般般吧,没什么特别的感觉",
    "收到货了,包装完整,还没开始使用",
    "价格还行,外观普通,功能中规中矩",
    "物流速度正常,三天到的,可以接受",
    "跟之前买的差不多,没有惊喜也没有失望",
    "东西就那样,一分钱一分货吧",
    "客服回复有点慢,但问题最终解决了",
    "说明书内容简单,还需要自己慢慢研究",
    "尺寸和描述一致,颜色略有差异",
    "先用一段时间看看,之后再追评",
    # 正面 2
    "物流速度很快,客服态度也很好,满意的一次购物",
    "性价比很高,推荐给朋友了,下次还会购买",
    "质量超出预期,做工精致,非常满意",
    "包装严实,发货迅速,宝贝和图片一样好看",
    "用了半个月很稳定,续航给力,五星好评",
    "客服专业耐心,问题秒回解决,点赞",
    "操作简单,老人也会用,家人都说买得值",
    "音质不错,外观大气,同事都问链接",
    "安装方便,效果明显,已经回购第二次了",
    "大品牌值得信赖,售后有保障,放心买",
]
labels = [0]*10 + [1]*10 + [2]*10      # 0=负面 1=中性 2=正面

# 1. 加载停用词表:有文件用文件,没有则用内置常用停用词
if os.path.exists("stopwords.txt"):
    with open("stopwords.txt", encoding="utf-8") as f:
        stopwords = set(f.read().split())
else:
    stopwords = set("的 了 是 我 你 他 她 它 也 都 就 在 和 与 及 或 啊 吧 呢 吗 "
                    "很 太 还 又 再 不 没 有 个 这 那 之 其 被 把 让 给 对 从 向 "
                    "一 二 三 个 次 天 些 上 下 里 中 时 时候 已经 一个 没有 这个 那个".split())
    print("未找到 stopwords.txt, 已使用内置停用词表(共%d个)" % len(stopwords))

# 2. 分词 -> 去停用词和长度<2的词 -> 空格连接
def clean_text(text):
    words = jieba.cut(text)   # 精确模式分词
    words = [w for w in words if w not in stopwords and len(w) >= 2]
    return " ".join(words)

cleaned = [clean_text(t) for t in texts]

# 3. 打印前5条 预处理前/后 对比
for i in range(5):
    print(f"原文: {texts[i]}")
    print(f"清洗: {cleaned[i]}\n")


FileNotFoundError: [Errno 2] No such file or directory: 'stopwords.txt'

## C2 TF-IDF 特征提取（2分）
**做什么**：`TfidfVectorizer(max_features=5000, ngram_range=(1,2))` → 训练集 `fit_transform`，测试集只 `transform` → 输出矩阵形状。

**记忆点**：训练集 fit+transform；测试集只 transform（共用同一词表，防止信息泄漏）。

In [ ]:
# ===== C2：TF-IDF特征提取 =====
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# 1. 定义向量化器:最多5000个特征,同时提取一元和二元词组
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# 2. 划分训练/测试文本(8:2)
X_tr_txt, X_te_txt, y_train, y_test = train_test_split(
    cleaned, labels, test_size=0.2, random_state=42, stratify=labels)

# 3. 训练集拟合+转换,测试集只转换;全量特征留给C8/C9聚类用
X_train_tfidf = tfidf.fit_transform(X_tr_txt)
X_test_tfidf = tfidf.transform(X_te_txt)
X_tfidf = tfidf.transform(cleaned)

print('训练集特征矩阵:', X_train_tfidf.shape)
print('测试集特征矩阵:', X_test_tfidf.shape)

## C3 传统机器学习情感分类（3分）
**做什么**：`SVC(kernel='rbf', C=1.0)` 训练 → predict 预测 → `accuracy_score` 算准确率。

**记忆点**：SVM三行搞定：`fit` 训练、`predict` 预测、`accuracy_score` 评分。

In [ ]:
# ===== C3：SVM情感分类 =====
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1. 训练SVM(RBF核, C=1.0)
svm = SVC(kernel='rbf', C=1.0)
svm.fit(X_train_tfidf, y_train)

# 2. 预测测试集并计算准确率
y_pred = svm.predict(X_test_tfidf)
acc = accuracy_score(y_test, y_pred)
print(f'SVM 测试集准确率: {acc:.4f}')

## C4 分类评估指标计算（3分）
**做什么**：`classification_report` 输出精确率/召回率/F1 → `confusion_matrix` → seaborn热力图（标签：负面/中性/正面）。

**记忆点**：报告看三类指标；热力图对角线越深越好。

In [ ]:
# ===== C4：分类评估指标与混淆矩阵 =====
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

names = ['负面', '中性', '正面']

# 1. 详细评估报告:精确率/召回率/F1
print(classification_report(y_test, y_pred, target_names=names))

# 2. 混淆矩阵热力图
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=names, yticklabels=names)
plt.xlabel('预测标签')
plt.ylabel('真实标签')
plt.title('情感分类混淆矩阵')
plt.tight_layout()
plt.show()

## C5 文本序列化与填充（4分）
**做什么**：`Tokenizer(num_words=10000)` 拟合 → `texts_to_sequences` 转整数序列 → `pad_sequences(maxlen=100, padding='post')` 填充 → 输出形状。

**记忆点**：词→数字→补齐长度，深度学习文本输入三步骤。

In [ ]:
# ===== C5：文本序列化与填充 =====
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. 分词器:只保留最高频的10000个词
tokenizer = Tokenizer(num_words=10000, oov_token='<OOV>')
tokenizer.fit_on_texts(cleaned)

# 2. 文本 -> 整数序列
seqs = tokenizer.texts_to_sequences(cleaned)

# 3. 填充到统一长度100,末尾补0
padded = pad_sequences(seqs, maxlen=100, padding='post')

print('填充后序列形状:', padded.shape)
print('实际词表大小:', min(len(tokenizer.word_index) + 1, 10000))

## C6 LSTM情感分析模型构建（4分）
**做什么**：Embedding(10000→128) → LSTM(128) → Dropout(0.5) → Dense(64,relu) → Dense(3,softmax)。

**记忆点**：Embedding把词索引变向量；LSTM处理序列；3单元softmax对应 负/中/正 三类。

In [ ]:
# ===== C6：LSTM情感分析模型 =====
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense

lstm_model = Sequential([
    Embedding(input_dim=10000, output_dim=128, input_length=100),
    LSTM(128, return_sequences=False),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')    # 负面/中性/正面 三分类
])
lstm_model.summary()

## C7 LSTM模型训练与评估（6分）
**做什么**：compile(Adam + sparse交叉熵) → fit(epochs=10, batch_size=64, validation_split=0.2) → 画准确率/损失双曲线。

**记忆点**：标签是整数用 `sparse_categorical_crossentropy`；双曲线看是否过拟合。

In [ ]:
# ===== C7：LSTM模型训练与评估 =====
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers import Adam

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

y = np.array(labels)

# 1. 编译:Adam(0.001) + 整数标签交叉熵 + 准确率
lstm_model.compile(optimizer=Adam(learning_rate=0.001),
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])

# 2. 训练
history = lstm_model.fit(padded, y,
                         epochs=10,
                         batch_size=64,
                         validation_split=0.2)

# 3. 绘制准确率与损失曲线
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history.history['accuracy'], label='训练')
ax[0].plot(history.history['val_accuracy'], label='验证')
ax[0].set_title('准确率曲线')
ax[0].legend()
ax[1].plot(history.history['loss'], label='训练')
ax[1].plot(history.history['val_loss'], label='验证')
ax[1].set_title('损失曲线')
ax[1].legend()
plt.tight_layout()
plt.show()

## C8 K-Means热点话题聚类（3分）
**做什么**：TF-IDF特征 → `KMeans(n_clusters=5)` 聚类 → 每类取中心权重最高的前10个词作话题关键词 → 词云展示。

**记忆点**：聚类中心 `cluster_centers_` 每维对应一个词的权重，权重最大的词就是话题代表。

In [ ]:
# ===== C8：K-Means热点话题聚类 =====
import os
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from wordcloud import WordCloud

plt.rcParams['font.sans-serif'] = ['SimHei']

# 1. 对TF-IDF特征做K-Means聚类
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_tfidf)

# 2. 每个聚类取中心权重Top10的词作为话题关键词
feature_names = tfidf.get_feature_names_out()
for c in range(5):
    top10 = kmeans.cluster_centers_[c].argsort()[::-1][:10]
    print(f'话题{c}: {[feature_names[i] for i in top10]}')

# 3. 为每个聚类生成词云
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for c in range(5):
    center = kmeans.cluster_centers_[c]
    freqs = {feature_names[i]: center[i]
             for i in center.argsort()[::-1][:50]}
    # 字体路径优先用Windows自带黑体,找不到则用当前目录simhei.ttf
    _font = "C:/Windows/Fonts/simhei.ttf" if os.path.exists("C:/Windows/Fonts/simhei.ttf") else "simhei.ttf"
    wc = WordCloud(font_path=_font,
                   background_color='white').generate_from_frequencies(freqs)
    axes[c].imshow(wc)
    axes[c].set_title(f'话题{c}')
    axes[c].axis('off')
plt.tight_layout()
plt.show()

## C9 肘部法则确定最佳聚类数（3分）
**做什么**：K从1到10循环聚类记录惯性 `inertia_` → 画K-惯性折线图 → 拐点（肘部）即最佳K。

**记忆点**：惯性下降由快变缓的拐点 = 最佳K；K再大收益很小。

In [ ]:
# ===== C9：肘部法则确定最佳K值 =====
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 1. K从1到10,记录每个K的惯性(簇内平方和)
inertias = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_tfidf)
    inertias.append(km.inertia_)

# 2. 绘制K-惯性折线图
plt.figure(figsize=(6, 4))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('聚类数 K')
plt.ylabel('惯性 inertia')
plt.title('肘部法则选择最佳K')
plt.xticks(list(K_range))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 3. 惯性下降明显变缓的拐点即最佳K
best_k = 3   # 根据图中肘部位置填写
print(f'最佳聚类数 K={best_k}: K>={best_k} 后惯性下降明显变缓, 继续增大K收益很小')